<p><font size="6" color='grey'> <b>
KI-Agenten. Verstehen. Anwenden. Gestalten.
</b></font> </br></p>


<p><font size="5" color='grey'> <b>
Multi-Tool Agents
</b></font> </br></p>

---

In [ ]:
#@title 🔧 Umgebung einrichten{ display-mode: "form" }
!uv pip install --system -q git+https://github.com/ralf-42/Agenten.git#subdirectory=04_modul

# LangSmith Env-Vars VOR allen LangChain-Imports setzen
import os
os.environ["LANGSMITH_TRACING"] = "true"
os.environ["LANGSMITH_PROJECT"]    = "M06-Multi-Tool-Agents"
os.environ["LANGSMITH_ENDPOINT"]   = "https://eu.api.smith.langchain.com"

from genai_lib.utilities import (
    check_environment,
    get_ipinfo,
    setup_api_keys,
    mprint,
    install_packages,
    mermaid,
    get_model_profile,
    extract_thinking,
    load_prompt,
    show_trace
)
setup_api_keys(['OPENAI_API_KEY', 'LANGSMITH_API_KEY'], create_globals=False)
print()
check_environment()
print()
get_ipinfo()


# 1 | Übersicht
---



In *Erste Agenten mit LangChain* wurde in erster Agenten mit **2 Tools** gebaut. Dieses Modul geht einen Schritt weiter:
**4 Tools**, komplexere Aufgaben und ein tiefes Verständnis, wie das LLM entscheidet, welches Tool es einsetzt.

| Merkmal | M03 – Einfacher Agent | M07 – Multi-Tool Agent |
|---|---|---|
| **Anzahl Tools** | 2 | 4 |
| **Aufgabentyp** | Einzel-Tool-Anfragen | Multi-Step-Anfragen |
| **Lernfokus** | Wie funktioniert ein Agent? | Warum wählt er Tool X? |
| **Debugging** | Direkte Ausgabe | LangSmith Trace |


In [ ]:
from langchain.chat_models import init_chat_model

# ── Konfigurationskonstante ────────────────────────────────────────────────
MAX_TOOL_CALLS = 10  # Maximale Tool-Aufrufe pro Agent-Lauf (recursion_limit)

llm = init_chat_model("openai:gpt-4o-mini", temperature=0.0)

In [ ]:
#@markdown   <p><font size="4" color='green'>  Prozessdiagramm</font> </br></p>


diagram = """
flowchart TB
    subgraph EINFACH["M03 – Einfacher Agent"]
        direction LR
        F1[Anfrage] --> LLM1[LLM]
        LLM1 -->|ein Tool| T1A[Tool 1 oder 2]
        T1A --> ANT1[Antwort]
    end
    subgraph MULTI["M07 – Multi-Tool Agent"]
        direction LR
        F2[Anfrage] --> LLM2[LLM]
        LLM2 -->|Schritt 1| T2A[Tool 1]
        T2A -->|Ergebnis 1| LLM2
        LLM2 -->|Schritt 2| T2B[Tool 2]
        T2B -->|Ergebnis 2| LLM2
        LLM2 --> ANT2[Antwort]
    end
"""
mermaid(diagram, width=1000)


# 2 | Agent mit 4 Tools
---



Jedes Tool übernimmt **eine klar abgegrenzte Aufgabe** aus einem anderen Fachbereich.
So kann der Agent mit einer einzigen Anfrage mehrere Themen bedienen.

| Tool | Domäne | Aufgabe |
|---|---|---|
| `celsius_nach_fahrenheit` | Temperatur | Einheitenumrechnung |
| `waehrung_umrechnen` | Finanzen | EUR, USD, GBP, CHF, JPY |
| `arbeitstage_berechnen` | Zeit / Datum | Datum nach N Arbeitstagen |
| `aufgabe_priorisieren` | Entscheidung | Eisenhower-Matrix |

**Designregeln für gute Tools:**
1. **Eine Verantwortung** – jedes Tool löst genau ein Problem
2. **Präziser Docstring** – das LLM kennt das Tool nur über seine Beschreibung
3. **Sprechende Parameternamen** – `betrag` statt `x`, `von_waehrung` statt `y`
4. **Klare Rückgabe** – Fehlermeldungen als String, kein `raise`


In [ ]:
#@markdown   <p><font size="4" color='green'>  Tool-Übersicht</font> </br></p>

diagram = """
flowchart LR
    AGENT["Agent\ngpt-4o-mini"]
    T1["🌡️ celsius_nach_fahrenheit\nTemperatur"]
    T2["💱 waehrung_umrechnen\nFinanzen"]
    T3["📅 arbeitstage_berechnen\nDatum"]
    T4["✅ aufgabe_priorisieren\nEisenhower-Matrix"]

    AGENT --> T1
    AGENT --> T2
    AGENT --> T3
    AGENT --> T4
"""
mermaid(diagram, width=650)


In [ ]:
from langchain_core.tools import tool
from datetime import date, timedelta

@tool
def celsius_nach_fahrenheit(temperatur: float) -> float:
    """Rechnet eine Temperatur von Celsius in Fahrenheit um."""
    return round(temperatur * 9 / 5 + 32, 2)

@tool
def waehrung_umrechnen(betrag: float, von_waehrung: str, nach_waehrung: str) -> str:
    """Rechnet einen Geldbetrag zwischen zwei Währungen um.
    Unterstützte Währungen: EUR, USD, GBP, CHF, JPY.
    betrag: Zu konvertierender Betrag.
    von_waehrung: Quellwährung als 3-Buchstaben-Code, z.B. EUR.
    nach_waehrung: Zielwährung als 3-Buchstaben-Code, z.B. USD."""
    kurse = {
        ("EUR", "USD"): 1.08, ("USD", "EUR"): 0.93,
        ("EUR", "GBP"): 0.86, ("GBP", "EUR"): 1.16,
        ("EUR", "CHF"): 0.94, ("CHF", "EUR"): 1.06,
        ("USD", "GBP"): 0.79, ("GBP", "USD"): 1.27,
        ("USD", "CHF"): 0.87, ("CHF", "USD"): 1.15,
    }
    von, nach = von_waehrung.upper(), nach_waehrung.upper()
    if von == nach:
        return f"{betrag:.2f} {nach}"
    kurs = kurse.get((von, nach))
    if not kurs:
        return f"Kurs für {von} → {nach} nicht verfügbar."
    return f"{betrag:.2f} {von} = {betrag * kurs:.2f} {nach}"

@tool
def arbeitstage_berechnen(start_datum: str, anzahl_arbeitstage: int) -> str:
    """Berechnet das Datum nach einer bestimmten Anzahl Arbeitstage (Montag–Freitag).
    start_datum: Startdatum im Format YYYY-MM-DD.
    anzahl_arbeitstage: Anzahl der Arbeitstage, die addiert werden sollen."""
    try:
        start = date.fromisoformat(start_datum)
    except ValueError:
        return f"Ungültiges Datum '{start_datum}'. Erwartet: YYYY-MM-DD"
    aktuell, gezaehlt = start, 0
    while gezaehlt < anzahl_arbeitstage:
        aktuell += timedelta(days=1)
        if aktuell.weekday() < 5:
            gezaehlt += 1
    tage = ["Montag","Dienstag","Mittwoch","Donnerstag","Freitag","Samstag","Sonntag"]
    return f"{anzahl_arbeitstage} Arbeitstage nach {start_datum}: {aktuell.isoformat()} ({tage[aktuell.weekday()]})"

@tool
def aufgabe_priorisieren(wichtigkeit: int, dringlichkeit: int) -> str:
    """Bewertet eine Aufgabe nach der Eisenhower-Matrix.
    wichtigkeit: Strategische Bedeutung der Aufgabe (1=gering bis 5=sehr hoch).
    dringlichkeit: Zeitlicher Druck (1=gering bis 5=sehr hoch)."""
    if not (1 <= wichtigkeit <= 5 and 1 <= dringlichkeit <= 5):
        return "Fehler: beide Werte müssen zwischen 1 und 5 liegen."
    if wichtigkeit >= 4 and dringlichkeit >= 4:
        return "Priorität A – Sofort erledigen (wichtig & dringend)"
    elif wichtigkeit >= 4:
        return "Priorität B – Termin setzen (wichtig, nicht dringend)"
    elif dringlichkeit >= 4:
        return "Priorität C – Delegieren (dringend, nicht wichtig)"
    else:
        return "Priorität D – Eliminieren oder verschieben"

# ── Think Tool (Strategische Reflexion zwischen Tool-Calls) ───────────────
# Bei Multi-Tool-Anfragen: Agent reflektiert nach jedem Tool-Call,
# bevor er das nächste Tool aufruft. Verhindert vorschnelle Folgecalls.
@tool
def think(reflection: str) -> str:
    """Strategisches Nachdenken zwischen Tool-Calls.

    Nutze dieses Tool nach einem Tool-Ergebnis:
    - Habe ich die Teilfrage vollständig beantwortet?
    - Welches Tool brauche ich als nächstes?
    - Sind alle nötigen Informationen für den nächsten Schritt vorhanden?

    Args:
        reflection: Analyse des aktuellen Stands und geplanter nächster Schritt
    """
    return f"Reflexion notiert: {reflection}"

mprint("✅ 5 Tools definiert:")
mprint("   celsius_nach_fahrenheit · waehrung_umrechnen · arbeitstage_berechnen")
mprint("   aufgabe_priorisieren · think (strategische Reflexion)")

In [ ]:
from langchain.agents import create_agent

tools = [celsius_nach_fahrenheit, waehrung_umrechnen, arbeitstage_berechnen, aufgabe_priorisieren, think]
system_prompt = load_prompt("https://github.com/ralf-42/Agenten/blob/main/05_prompt/m06_multi_tool_system_prompt.md", mode="S")

agent = create_agent(
    model="openai:gpt-4o-mini",
    tools=tools,
    system_prompt=system_prompt,
)

# Erster Test: einzel-Tool
antwort = agent.invoke({
    "messages": [{"role": "user", "content": "Was sind 25 Grad Celsius in Fahrenheit?"}]
})
mprint("## Erster Test")
mprint(antwort["messages"][-1].content)

# 3 | Komplexe Multi-Step Aufgaben
---

Bei **Multi-Step-Aufgaben** entscheidet das LLM schrittweise, welches Tool als nächstes aufgerufen wird. Ein Agent oder ein Framework steuert dabei die Ausführungsschleife, übergibt Zwischenergebnisse als Kontext und sorgt dafür, dass der nächste Tool-Call erst nach Vorliegen des vorherigen Ergebnisses erfolgt.


> *Das LLM entscheidet lokal – der Agent organisiert global.*

**Typische Multi-Step-Muster:**

| Muster | Beispiel |
|---|---|
| **Parallel**: unabhängige Tools | "Temperatur UND Währung umrechnen" |
| **Sequenziell**: Ergebnis 1 → Input 2 | Datum berechnen, Priorität daraus ableiten |
| **Bedingt**: nur wenn nötig | Agent entscheidet selbst, ob ein Tool nötig ist |

> 💡 **Beobachten:** Zählen Sie die Tool-Calls im nächsten Beispiel – wie viele braucht der Agent?


In [ ]:
#@markdown   <p><font size="4" color='green'>  Ablauf einer Multi-Step Anfrage</font> </br></p>

diagram = """
sequenceDiagram
    autonumber
    participant User
    participant Agent
    participant LLM as LLM (gpt-4o-mini)
    participant W as waehrung_umrechnen
    participant A as arbeitstage_berechnen

    User->>Agent: "500 EUR in USD? Abgabe in 10 Arbeitstagen ab heute?"
    Agent->>LLM: Nutzeranfrage + Tool-Schemas + aktuelles Datum
    LLM->>Agent: Tool-Call: waehrung_umrechnen
    Agent->>W: betrag=500, von=EUR, nach=USD
    W->>Agent: "500.00 EUR = 540.00 USD"
    Agent->>LLM: Tool-Ergebnis: Währungsumrechnung
    LLM->>Agent: Tool-Call: arbeitstage_berechnen
    Agent->>A: start=heutiges_Datum, anzahl=10
    A->>Agent: "10 Arbeitstage ab heute: 2026-04-01"
    Agent->>LLM: Tool-Ergebnis: Arbeitstage berechnet
    LLM->>Agent: Finale Antwort formulieren
    Agent->>User: Antwort mit beiden Ergebnissen
"""
mermaid(diagram, width=1100)


In [ ]:
# Komplexe Anfrage, die zwei Tools benötigt
komplexe_anfrage = (
    "Zwei Infos für unser Projektmeeting: "
    "1. Was sind 500 EUR in USD? "
    "2. Wenn wir heute (2026-03-03) starten und 10 Arbeitstage Zeit haben – wann ist der Abgabetermin?"
)

ergebnis_multi = agent.invoke({
    "messages": [{"role": "user", "content": komplexe_anfrage}]
})

mprint("## Multi-Step Ergebnis")
mprint(ergebnis_multi["messages"][-1].content)


In [ ]:
mprint("## Tool-Calls in dieser Anfrage")
schritt = 1
for msg in ergebnis_multi["messages"]:
    if hasattr(msg, "tool_calls") and msg.tool_calls:
        for tc in msg.tool_calls:
            mprint(f"**Schritt {schritt}:** `{tc['name']}` mit Args `{tc['args']}`")
            schritt += 1
if schritt == 1:
    mprint("*Kein Tool-Call – direkte LLM-Antwort*")


# 4 | Agent-Debugging mit LangSmith
---

Bei Multi-Tool Agenten wird LangSmith besonders wertvoll: Der vollständige TAO-Zyklus
(Think – Act – Observe) ist im Trace sichtbar – **für jeden einzelnen Tool-Call**.

**Was man im Multi-Tool Trace sieht:**

| Trace-Element | Was es zeigt |
|---|---|
| **LLM-Call (Think)** | Wie das LLM die Anfrage und bisherigen Ergebnisse verarbeitet |
| **Tool-Call (Act)** | Welches Tool mit welchen Argumenten aufgerufen wurde |
| **Tool-Response (Observe)** | Was das Tool zurückgegeben hat |
| **Iterationen** | Wie oft der TAO-Zyklus wiederholt wurde |
| **Token-Verbrauch** | Gesamt und pro Schritt |

> 💡 **Tipp:** Öffnen Sie LangSmith parallel und beobachten Sie, wie der Agent durch
> die Zwischenschritte navigiert. Der Trace zeigt mehr als die finale Antwort.


In [ ]:
# 1. Tracing-Konfiguration vorab festlegen
run_cfg = {
    "run_name": "M06_Kap4_MultiToolTrace",
    "tags":     ["m06", "multi-tool", "langsmith"]
}

# 2. with_config() anwenden, dann invoke()
trace_anfrage = (
    "Ich plane eine USA-Reise: Was sind 800 EUR in USD? "
    "Und: Heute ist 2026-03-03 – wann ist in 5 Arbeitstagen?"
)

trace_ergebnis = agent.with_config(**run_cfg).invoke({
    "messages": [{"role": "user", "content": trace_anfrage}]
})

mprint("## Trace-Ergebnis")
mprint(trace_ergebnis["messages"][-1].content)

In [ ]:
# usage_metadata direkt aus der letzten AI-Message auslesen
# (aus dem Multi-Step Beispiel: trace_ergebnis)
letzte_ai_msg = trace_ergebnis["messages"][-1]
usage = getattr(letzte_ai_msg, "usage_metadata", None)

if usage:
    inp  = usage.get("input_tokens",  0)
    out  = usage.get("output_tokens", 0)
    tot  = usage.get("total_tokens",  0)
    # gpt-4o-mini Preise (Richtwert Stand 2026, pro 1M Token)
    cost_in  = inp * 0.15   / 1_000_000
    cost_out = out * 0.60   / 1_000_000
    cost     = cost_in + cost_out
    print(
        f"Token-Verbrauch (letzter Trace-Run)\n"
        f"{'-'*40}\n"
        f"Input : {inp:>6} Tokens | ${cost_in:.5f}\n"
        f"Output: {out:>6} Tokens | ${cost_out:.5f}\n"
        f"{'-'*40}\n"
        f"Total : {tot:>6} Tokens | ${cost:.5f}\n\n"
        f"Preisbasis: gpt-4o-mini ($0.15 / $0.60 per 1M Token)\n"
    )
else:
    print("Keine usage_metadata in dieser Message (Tool-Messages geben kein usage_metadata zurueck).")
    print("Tipp: usage_metadata ist nur an AIMessages verfuegbar.")

## Token- und Kostenueberblick

Jeder Agent-Aufruf verbraucht Tokens – bei Multi-Step-Anfragen summiert sich das ueber mehrere LLM-Calls hinweg. `usage_metadata` macht den Verbrauch direkt messbar.

> **LangSmith zeigt Kosten pro Run** – aber `usage_metadata` ermoeglicht auch schnelle In-Notebook-Schaetzungen ohne Dashboard-Wechsel.

# 5 | Warum wählt der Agent Tool X?
---

Das LLM kennt die Tools **ausschließlich über ihre Docstrings**.
Die Beschreibung ist der einzige Kompass für die Tool-Auswahl.

**Was passiert bei der Tool-Auswahl:**
1. Das LLM liest alle Tool-Beschreibungen
2. Es wählt das Tool, dessen Beschreibung am besten zur Anfrage passt
3. Es generiert die Argumente passend zu den Parameternamen

> ⚠️ **Konsequenz:** Ein vager Docstring führt zu falschen Tool-Calls oder dazu,
> dass der Agent ein Tool gar nicht aufruft – obwohl es die richtige Antwort hätte.

Im folgenden Experiment vergleichen wir zwei Varianten desselben Tools:
eine mit **vager** und eine mit **präziser** Beschreibung.


In [ ]:
#@markdown   <p><font size="4" color='green'>  Tool-Auswahlprozess des LLM</font> </br></p>

diagram = """
flowchart TB
    FRAGE["Anfrage: 'Was sind 30 Grad Celsius in Fahrenheit?'"]
    LLM["LLM vergleicht\nAnfrage mit Tool-Docstrings"]
    T1["celsius_nach_fahrenheit\n✅ 'Rechnet Celsius in Fahrenheit'"]
    T2["waehrung_umrechnen\n❌ 'Geldbetrag umrechnen'"]
    T3["arbeitstage_berechnen\n❌ 'Datum berechnen'"]
    T4["aufgabe_priorisieren\n❌ 'Aufgabe bewerten'"]
    AUSWAHL["Auswahl: celsius_nach_fahrenheit\nArg: temperatur=30.0"]

    FRAGE --> LLM
    LLM --> T1 & T2 & T3 & T4
    T1 --> AUSWAHL

    style T1 fill:#d5e8d4
    style T2 fill:#f8cecc
    style T3 fill:#f8cecc
    style T4 fill:#f8cecc
    style AUSWAHL fill:#dae8fc
"""
mermaid(diagram, width=800)


In [ ]:
@tool
def umrechner_vage(x: float, y: str, z: str) -> str:
    """Rechnet Werte um."""
    kurse = {("EUR", "USD"): 1.08, ("USD", "EUR"): 0.93,
             ("EUR", "GBP"): 0.86, ("GBP", "EUR"): 1.16}
    kurs = kurse.get((y.upper(), z.upper()))
    return f"{x * kurs:.2f} {z.upper()}" if kurs else "Kurs nicht verfuegbar"

@tool
def umrechner_praezise(betrag: float, von_waehrung: str, nach_waehrung: str) -> str:
    """Rechnet einen Geldbetrag von einer Waehrung in eine andere um.
    von_waehrung: Quellwaehrung als Code (EUR, USD oder GBP).
    nach_waehrung: Zielwaehrung als Code (EUR, USD oder GBP)."""
    kurse = {("EUR", "USD"): 1.08, ("USD", "EUR"): 0.93,
             ("EUR", "GBP"): 0.86, ("GBP", "EUR"): 1.16}
    kurs = kurse.get((von_waehrung.upper(), nach_waehrung.upper()))
    return f"{betrag * kurs:.2f} {nach_waehrung.upper()}" if kurs else "Kurs nicht verfuegbar"

assistent_prompt = "Du bist ein Assistent. Antworte auf Deutsch."
agent_vage = create_agent(model="openai:gpt-4o-mini", tools=[umrechner_vage], system_prompt=assistent_prompt)
agent_prae = create_agent(model="openai:gpt-4o-mini", tools=[umrechner_praezise], system_prompt=assistent_prompt)

testfrage = "Rechne 200 Euro in Britische Pfund um."

r_vage = agent_vage.invoke({"messages": [{"role": "user", "content": testfrage}]})
r_prae = agent_prae.invoke({"messages": [{"role": "user", "content": testfrage}]})

mprint("## Ergebnis: Vage Beschreibung (`'Rechnet Werte um.'`)")
mprint(r_vage["messages"][-1].content)
mprint("## Ergebnis: Praezise Beschreibung")
mprint(r_prae["messages"][-1].content)


# 6 | Error Handling & Fallbacks
---

Wenn ein Tool einen Fehler wirft (`raise`), bricht der Agent in der Regel ab.
Besser: Das Tool **gibt einen Fehler als String zurück** – dann kann das LLM
die Fehlermeldung lesen, erklären und den User um eine Korrektur bitten.

| Strategie | Beschreibung | Empfehlung |
|---|---|---|
| `raise Exception` | Tool wirft Fehler | ❌ Agent bricht ab |
| `return "Fehler: ..."` | Tool gibt Fehlermeldung zurück | ✅ Agent erklärt |
| Validierung vor Logik | Werte prüfen, bevor gerechnet wird | ✅ Frühzeitig fangen |

**Typische Fehlerquellen:**
- Ungültige Datumsformate (`"morgen"` statt `"2026-03-03"`)
- Werte außerhalb erlaubter Bereiche (z.B. `wichtigkeit=7`)
- Währungspaare ohne Kursdaten (z.B. JPY → GBP)
- Division durch Null


In [ ]:
# Tool mit robuster Fehlerbehandlung - return statt raise
@tool
def sichere_division(dividend: float, divisor: float) -> str:
    """Dividiert zwei Zahlen sicher. Gibt Fehlermeldung bei Division durch Null zurueck.
    dividend: Zaehler.
    divisor: Nenner - darf nicht 0 sein."""
    if divisor == 0:
        return "Fehler: Division durch Null nicht moeglich. Bitte einen Nenner != 0 angeben."
    return f"{dividend} ÷ {divisor} = {dividend / divisor:.4f}"

robust_prompt = load_prompt("https://github.com/ralf-42/Agenten/blob/main/05_prompt/m06_robust_rechenassistent_system_prompt.md", mode="S")
agent_robust = create_agent(
    model="openai:gpt-4o-mini",
    tools=[sichere_division, waehrung_umrechnen],
    system_prompt=robust_prompt,
)

# Fehlerfall: Division durch Null
r_fehler = agent_robust.invoke({
    "messages": [{"role": "user", "content": "Teile 100 durch 0."}]
})
mprint("## Fehlerfall: Division durch Null")
mprint(r_fehler["messages"][-1].content)


In [ ]:
# Kombination: Nach Fehler direkt weiter mit gültiger Anfrage
r_kombi = agent_robust.invoke({
    "messages": [
        {"role": "user",      "content": "Teile bitte 50 durch 0."},
        {"role": "assistant", "content": r_fehler["messages"][-1].content},
        {"role": "user",      "content": "Ok, dann durch 4. Und: Was sind 300 USD in EUR?"}
    ]
})
mprint("## Robuste Fortsetzung nach Fehler")
mprint(r_kombi["messages"][-1].content)


In [ ]:
#@markdown   <p><font size="4" color='green'>  LangSmith Trace-Analyse</font> </br></p>

import time as _t; _t.sleep(2)
show_trace("M06-Multi-Tool-Agents", limit=3, show_steps=True)

# A | Aufgabe
---

<p><font color='darkblue' size="4">
📌 <b>Wichtig</b>
</font></p>

Die Aufgabestellungen unten bieten Anregungen, Sie können aber auch gerne eine andere Herausforderung angehen.

**Hinweis zur Lösungshilfe:**
> In diesem Kurs dürfen und sollen Sie generative KI auch als Unterstützung beim Lernen und Entwickeln nutzen. Wenn Sie bei einer Aufgabe festhängen, können Sie zum Beispiel Gemini in Google Colab verwenden, um Fehlermeldungen besser zu verstehen, Ideen für Teilschritte zu bekommen oder Code-Varianten zu prüfen.
> <br>**Wichtig ist nur:** Die KI dient als Lern- und Entwicklungshilfe. Der Schwerpunkt des Kurses bleibt darauf, KI-Agenten selbst zu verstehen, aufzubauen und gezielt weiterzuentwickeln.


<p><font color='black' size="5">
Eigener Multi-Tool Agent
</font></p>

Bauen Sie einen eigenen Agenten für ein Szenario aus Ihrem Arbeitsumfeld –
mit mindestens 3 Tools aus unterschiedlichen Domänen.

**Mindest-Anforderungen:**
- Mindestens **3 selbst definierte Tools** aus verschiedenen Bereichen
- Präzise Docstrings mit deutschen Parameterbeschreibungen
- Einen passenden System-Prompt für die Rolle des Agenten
- Mindestens **2 Multi-Step Anfragen** (Agent nutzt ≥ 2 Tools pro Anfrage)
- **Fehlerbehandlung** in mindestens einem Tool (Rückgabe statt `raise`)



**Optionale Teilaufgaben:**
- Führen Sie das Beschreibungs-Experiment durch: Verschlechtern Sie absichtlich einen Docstring
  und beobachten Sie, wie sich das Verhalten des Agenten verändert
- Öffnen Sie LangSmith während einer Multi-Step Anfrage und zählen Sie die TAO-Zyklen
- Fügen Sie ein 4. Tool hinzu und testen Sie Anfragen, die alle 4 Tools benötigen
- Nutzen Sie `with_config()` und beobachten Sie den Run in LangSmith unter einem eigenen Namen


<p><font color='black' size="5">
🔍 Selbstcheck mit `assert`
</font></p>

`assert` prüft eine Bedingung — ist sie `False`, stoppt Python mit einem `AssertionError` und zeigt die Fehlermeldung an:

```python
assert bedingung, "Fehlermeldung"

assert 2 + 2 == 4, "Rechnung falsch"    # ✅ kein Fehler
assert len("Hi") > 5, "Text zu kurz"   # ❌ AssertionError: Text zu kurz
```

**So nutzen Sie den Selbstcheck:**
1. Schreiben Sie Ihre Lösung in die Zellen über diesem Block
2. Speichern Sie Ihren Agenten in **`mein_agent`** und Ihre Tools-Liste in **`meine_tools`**
3. Führen Sie die Zelle unten aus — ein `✅` zeigt: Mindestanforderungen erfüllt


<p><font color='black' size="5">
✅ Selbstcheck
</font></p>

In [ ]:
# ► Speichere deinen Agenten in 'mein_agent' und deine Tools-Liste in 'meine_tools'.

_ag  = mein_agent   # ← Variablennamen anpassen
_tls = meine_tools  # ← Liste deiner Tool-Funktionen anpassen

assert hasattr(_ag, "invoke"), \
    "❌ Kein invoke() – wurde der Agent mit create_agent() erstellt?"
assert len(_tls) >= 3, \
    f"❌ Nur {len(_tls)} Tool(s) – Mindestanforderung: ≥ 3 Tools"

_r = _ag.invoke({"messages": [{"role": "user", "content": "Nutze deine Tools für eine kurze Demo."}]})
_msgs = _r["messages"]
_tool_calls = [m for m in _msgs if getattr(m, "type", None) == "tool"]
assert len(_tool_calls) >= 1, \
    "❌ Kein Tool-Aufruf erkannt – formuliere die Anfrage tool-spezifischer"

print(f"✅ {len(_tls)} Tools · {len(_tool_calls)} Aufruf(e) in diesem Test")
print(f"   {_msgs[-1].content[:120]}{'...' if len(_msgs[-1].content) > 120 else ''}")
